# Regression 2: Autograd og smarte optimizers

I regression 1 regnede I selv gradienten ud i hånden og tog ét lille skridt ned ad bakken ad gangen. Her bygger vi videre på to måder:

- **Autograd:** i stedet for at udlede gradienten selv, lader vi PyTorch finde den med `.backward()`. Så virker gradient descent på et hvilket som helst loss — også dem, I ikke kan differentiere i hånden.
- **Smarte optimizers:** når der er mange punkter (eller mange skridt), er der bedre måder at bevæge sig på end almindelig gradient descent. I bygger selv **minibatch/SGD**, **Momentum**, **RMSprop** og til sidst **ADAM** — og dyster med dem på fire landskaber.

Til allersidst får I en kort forsmag på **neurale netværk**, som trænes med præcis de samme værktøjer.


# 1: Gradienten med autograd

Vi starter med det samme datasæt som i regression 1 — bare med 100 punkter i stedet for 4. Loss er den samme (MSE for linjen $y = a\cdot x + b$), men gradienten regner vi ikke længere i hånden.

I stedet gør vi $a$ og $b$ til **tensorer**, som I kender fra intro til programmering, og beder PyTorch om gradienten med `.backward()`:


In [ ]:
# Henter hjælpefunktioner + testfil fra GitHub (Plan B: upload dem manuelt via mappeikonet i Colab)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/98-Helpers/helpers.py
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/02-Regression/regression2/regression2_test.py

import numpy as np
import helpers as sesy_viz          # sesy_vizualisation er foldet ind i helpers.py
import helpers as tk                # regression2_helper (LINJE_PUNKTER, LANDSKABER) er også i helpers.py

In [ ]:
import torch

points = tk.LINE_POINTS   # ~100 punkter, normalfordelt omkring en linje

# MSE for linjen y = a*x + b (samme loss som i regression 1)
def loss(a, b, points):
    n = len(points)
    total = 0
    for x, y in points:
        total += (a*x + b - y)**2
    return total / n

# gradienten af loss - nu med autograd i stedet for i hånden:
# gør a og b til tensorer, kald loss(a, b).backward(), og aflæs .grad
def gradient(a, b, points):
    a = torch.tensor(float(a), dtype=torch.float64, requires_grad=True)
    b = torch.tensor(float(b), dtype=torch.float64, requires_grad=True)
    loss(a, b, points).backward()
    return a.grad.item(), b.grad.item()

# ét gradient descent-skridt (som i regression 1)
def step(a, b, points, lr):
    da, db = gradient(a, b, points)
    return a - lr*da, b - lr*db

Her er en illustration af losslandskabet og af, hvordan gradient descent træner med alle 100 punkter:


In [ ]:

a, b = 0.0, 0.0
lr = 0.02

path = [(a, b)]
paneler = []
for _ in range(50):
    a, b = path[-1]
    grad_a, grad_b = gradient(a, b, points)
    paneler.append((
        sesy_viz.modelfit(a, b, points, x_range=(0,10), y_range=(-2,14)),
        sesy_viz.loss_contour(lambda a, b: loss(a, b, points), a_range=(-1,3), b_range=(-3,3), resolution=40, path=list(path), gradient=(grad_a, grad_b), colorbar=False),
        sesy_viz.loss_over_time([loss(a, b, points) for a, b in path]),
    ))
    path.append(step(a, b, points, lr))

print("Beregning færdig, samler til animation:")
sesy_viz.animate(paneler, max_frames=25)

# 2: Minibatch og stokastisk gradient descent

Hvorfor regne gradienten ud fra ALLE punkter for hvert skridt, hvis vi kan nøjes med et tilfældigt udpluk?

Med 100 punkter (og for rigtige datasæt ofte millioner) er det dyrt at bruge alle punkter hvert skridt. Løsningen er at trække et tilfældigt udpluk — en **batch** — og regne gradienten ud fra den. Den gradient peger næsten samme vej som gradienten fra alle punkter, men er meget billigere. Det kaldes **minibatch** eller **stokastisk gradient descent (SGD)**.

### Opgaver

##### Opgave 2.1

Skriv `opg1_1(punkter, k)`, der trækker `k` tilfældige punkter UDEN tilbagelægning fra `punkter`.

##### Opgave 2.2

Skriv træningsløkken `opg1_2`: træk en ny batch med `opg1_1`, og tag ét `skridt` (fra afsnit 1) — `n` gange i træk.


In [ ]:
from regression2_test import *
import random

# træk k tilfældige punkter, UDEN tilbagelægning, fra en liste af punkter
def ex1_1(points, k):
    return random.sample(points, k)
test_ex1_1(ex1_1)

# træningsloop: træk en ny batch (opg1_1) og tag ét skridt (skridt, fra recap), n gange i træk
def ex1_2(a, b, points, batch_size, lr, n):
    for _ in range(n):
        batch = ex1_1(points, batch_size)
        a, b = step(a, b, batch, lr)
    return a, b
test_ex1_2(ex1_2)

Illustrationen viser, hvordan loss-konturen ændrer sig med forskellige batch-størrelser, og hvordan SGD-stien ser ud:


In [ ]:
batch_size = 5   # prøv fx 1 (meget jitter) eller 50 (næsten roligt) og kør cellen igen

full_loss_fn = lambda a, b: loss(a, b, points)   # landskabet regnet ud fra ALLE punkter — stabilt facit-landskab

a, b = 0.0, 0.0
lr = 0.02

path = [(a, b)]
paneler = []
for _ in range(100):
    a, b = path[-1]
    batch = ex1_1(points, batch_size)             # jeres egen funktion
    grad_a, grad_b = gradient(a, b, batch)                # kun til at tegne pilen — selve skridtet tages nedenfor
    batch_loss_fn = lambda a, b, batch=batch: loss(a, b, batch)   # landskabet regnet KUN ud fra denne batch
    paneler.append((
        sesy_viz.modelfit(a, b, points, x_range=(0,10), y_range=(-2,14)),
        sesy_viz.loss_contour(full_loss_fn, a_range=(-1,3), b_range=(-3,3), resolution=40, path=list(path), gradient=(grad_a, grad_b), colorbar=False, title='loss(alle punkter)'),
        sesy_viz.modelfit(a, b, batch, x_range=(0,10), y_range=(-2,14)),
        sesy_viz.loss_contour(batch_loss_fn, a_range=(-1,3), b_range=(-3,3), resolution=40, path=list(path), gradient=(grad_a, grad_b), colorbar=False, title='loss(kun denne batch)'),
    ))
    path.append(step(a, b, batch, lr))   # skridt fra recap, PÅ SAMME batch

print("Beregning færdig, samler til animation:")
sesy_viz.animate(paneler, max_frames=50)

##### Opgave 2.3

Kør cellen ovenfor med forskellige batch-størrelser (1, 5, 20, 50, 100). *(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*


# 3: Smarte optimizers — en lille konkurrence

Kan vi komme hurtigere og mere direkte til minimum, uden at bruge flere punkter pr. skridt? Det er dét, metoder som **ADAM** — en af de mest brugte optimizers overhovedet — forsøger.

I stedet for at gennemgå ADAM slavisk, gør vi det til en lille konkurrence. I skriver en funktion `metode(loss, start, ...)`, der prøver at finde punktet med lavest loss.

**Reglerne:**
- I får kun `loss(a, b)` — ikke gradienten. Den finder I selv med `.backward()` (se `gradienten` nedenfor).
- I må kalde `loss` så mange gange, I vil, men vi stopper jer efter **100** kald tilsammen, så metoderne kan sammenlignes retfærdigt.
- I må gerne stoppe og returnere før budgettet er brugt, hvis I er tætte nok på.
- Bruger I alle 100 kald uden selv at returnere, bruges automatisk det sidste punkt, I nåede at undersøge (I får en advarsel).
- Beder I om et punkt meget langt fra landskabet (over 1000 væk), stopper det på samme måde — det er nok en fejl (fx et alt for stort skridt).

Sammenligningen køres på **4 landskaber**: `linjefitting` (som I kender) og 3 "underlige" landskaber, I ikke kender formlen for — og fra 4 forskellige startpunkter, valgt så metoderne opfører sig så forskelligt som muligt.

I bygger `gd_metode`, og siden `momentum_metode`, `rmsprop_metode` og `adam_metode` — hver i to lag: selve opdateringen (`..._skridt`, testet for sig) og løkken udenom (`..._metode`, testet som helhed). Værktøjer fra `tk`:
- `test_..._skridt(...)` og `test_..._metode(...)` tjekker, om jeres formel er rigtig.
- `tk.evaluer_gd_metode([("gd", gd_metode), ...])` viser metoderne side om side: et søjlediagram pr. landskab og hver metodes sti fra de 4 startpunkter. Skriv hele listen igen, hver gang I vil sammenligne.


In [ ]:
# I konkurrencen får I KUN loss(a, b) - ikke gradienten. Men den finder I selv med autograd:
# gør a og b til tensorer, kald loss(a, b).backward(), og aflæs .grad. Samme .backward()-trick
# som i afsnit 1, nu for et hvilket som helst loss.
def get_gradient(loss, a, b):
    a = torch.tensor(float(a), dtype=torch.float64, requires_grad=True)
    b = torch.tensor(float(b), dtype=torch.float64, requires_grad=True)
    loss(a, b).backward()
    return a.grad.item(), b.grad.item()

## Almindelig gradient descent

Den simpleste metode: find gradienten, og tag ét skridt imod den — igen og igen.


In [ ]:
from regression2_test import *

# ét gradient descent-skridt: find gradienten af loss med .backward(), og træk lr*gradienten fra
def gd_step(a, b, loss, lr):
    da, db = get_gradient(loss, a, b)
    return a - lr*da, b - lr*db
test_gd_step(gd_step)

# løkken udenom: kald gd_skridt n gange i træk
def gd_method(loss, start, lr=0.01, n=100):
    a, b = start
    for _ in range(n):
        a, b = gd_step(a, b, loss, lr)
    return a, b
test_gd_method(gd_method)

tk.evaluate_gd_method([("gd", gd_method)])

**Frit spil:** bemærk især hvor galt det går på `rosenbrock` — prøv at forbedre den, som
`egen_gd_metode_v2` (andet `lr`, eller en helt anden idé). Ingen facit at teste imod her —
tilføj jeres variant til listen og sammenlign direkte med `tk.evaluer_gd_metode(...)`:


In [ ]:
def own_gd_method_v1(loss, start, lr=0.02, n=100):
    a, b = start
    da, db = get_gradient(loss, a, b)
    for _ in range(n):
        a, b = a - lr * da, b - lr * db
    return a, b

tk.evaluate_gd_method([
    ("gd", gd_method),
    ("egen v1", own_gd_method_v1),
])

## Momentum

Momentum er IKKE en fysisk "bold der ruller ned ad bakken" — det er en **vægtet, aftagende
gennemsnit af alle de gradienter man har set indtil videre**:

$$v_t = (\beta) \cdot v_{t-1} + (1-\beta)\cdot \nabla L(a_t, b_t), \qquad (a,b)_{t+1} = (a,b)_t - \text{lr}\cdot v_t$$

$\beta$ (typisk omkring 0.9) styrer hvor meget vægt de ældre gradienter stadig har — en
gradient fra $k$ skridt tilbage tæller med vægten $(1-\beta)\beta^k$, altså mindre og mindre
jo længere tilbage den er. Bygges i to lag, ligesom `gd_metode`: `momentum_v_opdater` (selve
$v$-formlen) og `momentum_skridt` (ét skridt, som bruger $v$ i stedet for selve gradienten).

In [ ]:
from regression2_test import *

# opdater det glidende gennemsnit v, med den nye gradient vægtet ind
def momentum_v_update(va, vb, da, db, beta):
    return beta*va + (1-beta)*da, beta*vb + (1-beta)*db
test_momentum_v_update(momentum_v_update)

# ét momentum-skridt: find gradienten, opdater v, brug SÅ v (ikke selve gradienten) til skridtet
def momentum_step(a, b, va, vb, loss, lr, beta):
    da, db = get_gradient(loss, a, b)
    va, vb = momentum_v_update(va, vb, da, db, beta)
    return a - lr*va, b - lr*vb, va, vb
test_momentum_step(momentum_step)

# løkken udenom: kald momentum_skridt n gange i træk, og hold styr på v undervejs
def momentum_method(loss, start, lr=0.02, beta=0.9, n=99):
    a, b = start
    va, vb = 0.0, 0.0
    for _ in range(n):
        a, b, va, vb = momentum_step(a, b, va, vb, loss, lr, beta)
    return a, b
test_momentum_method(momentum_method)

Se momentum illustreret med jeres eget `momentum_skridt` kaldt for hvert billede: 
* venstre panel er gradienten lige nu (+ alle tidligere målinger, gråt), 
* midten samler dem til $v$ (ældre vejer mindre),
* højre er det skridt der rent faktisk tages:

In [ ]:
landscape = [l for l in tk.LANDSCAPES if l["name"] == "plateau (1D)"][0]
lr, beta = 0.1, 0.9
show_scale = 1.0   # ren visningsskala for pilene — juster hvis de er for små/store til at se

a, b = 1, 0.6
va, vb = 0.0, 0.0
references = []   # hver tidligere gradients bidrag (1-beta)*grad, DA den var ny
measurements = []     # (a, b, ref_a, ref_b) — hvor + hvad, til panel 1's grå pile
path = [(a, b)]

frames = []
for _ in range(100):
    da, db = get_gradient(landscape["loss"], a, b)   # kun til visning i panel 1 (selve skridtet tages nedenfor)
    reference = (show_scale * (1 - beta) * da, show_scale * (1 - beta) * db)
    references.append(reference)
    measurements.append((a, b, *reference))

    # ét par pr. tidligere gradient: (reference = værdien DA den kom) og (nu = dens aktuelle,
    # vægtede bidrag til v_t — reference skrumpet med beta**alder)
    n = len(references)
    pair = [(ref, (ref[0] * beta**alder, ref[1] * beta**alder))
           for alder, ref in zip(range(n - 1, -1, -1), references)]

    ny_a, ny_b, va, vb = momentum_step(a, b, va, vb, landscape["loss"], lr, beta)   # jeres egen funktion

    shared = dict(path=list(path), colorbar=False)
    frames.append((
        sesy_viz.loss_contour(landscape["loss"], landscape["a_range"], landscape["b_range"], gradient=(show_scale * da, show_scale * db),
                              extra_arrows=[(ma, mb, mda, mdb, "gray", 0.5) for ma, mb, mda, mdb in measurements],
                              title="gradient nu (+ tidligere målinger)", **shared),
        sesy_viz.vector_fan(pair, sum_vector=(show_scale * va, show_scale * vb), title="tidligere gradienter: original vs. vægtet"),
        sesy_viz.loss_contour(landscape["loss"], landscape["a_range"], landscape["b_range"], gradient=(show_scale * va, show_scale * vb),
                              gradient_color="orange", title="momentum-skridt", **shared),
    ))
    a, b = ny_a, ny_b
    path.append((a, b))

sesy_viz.animate(frames, max_frames=50)

**Frit spil:** prøv jeres egen variant af momentum — fx et andet `beta`, eller en helt anden
idé for hvordan I bruger de tidligere gradienter. Tilføj den til listen og sammenlign:

In [ ]:
def own_gd_method_v2(loss, start, lr=0.02, n=100):
    a, b = start
    da, db = get_gradient(loss, a, b)
    for _ in range(n):
        a, b = a - lr * da, b - lr * db
    return a, b

tk.evaluate_gd_method([
    ("gd", gd_method),
    ("momentum", momentum_method),
    ("egen v2", own_gd_method_v2),
])

## RMSprop

RMSprops idé: skalér hver akse med sin egen (nyligt sete) gradient-størrelse, så gradienten i den skalerede akse ligger omkring $\pm 1$ .
ie vi optimere ikke bare en værdi ad gangen, som nogen gange sker med den normale gradient.
dermed tages der lige store skridt i alle retninger, uanset om den oprindelige akse var stejl eller flad:

$$s_t = \beta\cdot s_{t-1}+(1-\beta)\cdot \nabla L(a_t,b_t)^2, \qquad (a,b)_{t+1} = (a,b)_t - \text{lr}\cdot\frac{\nabla L(a_t,b_t)}{\sqrt{s_t}+\varepsilon}$$

($s$ regnes ELEMENTVIS pr. parameter — $a$ og $b$ får hver deres egen skalering. $\varepsilon$
er blot et lille tal, der forhindrer division med 0.) Samme to-lags opskrift som momentum:
`rmsprop_s_opdater` (selve $s$-formlen) og `rmsprop_skridt` (ét skridt).

In [ ]:
from regression2_test import *

# opdater det glidende gennemsnit s af GRADIENTEN I ANDEN
def rmsprop_s_update(sa, sb, da, db, beta):
    return beta*sa + (1-beta)*da**2, beta*sb + (1-beta)*db**2
test_rmsprop_s_update(rmsprop_s_update)

# ét rmsprop-skridt: find gradienten, opdater s, og skalér gradienten med sqrt(s) før skridtet
def rmsprop_step(a, b, sa, sb, loss, lr, beta, eps):
    da, db = get_gradient(loss, a, b)
    sa, sb = rmsprop_s_update(sa, sb, da, db, beta)
    return a - lr*da/(sa**0.5+eps), b - lr*db/(sb**0.5+eps), sa, sb
test_rmsprop_step(rmsprop_step)

# løkken udenom: kald rmsprop_skridt n gange i træk, og hold styr på s undervejs
def rmsprop_method(loss, start, lr=0.02, beta=0.5, eps=1e-8, n=99):
    a, b = start
    sa, sb = 0.0, 0.0
    for _ in range(n):
        a, b, sa, sb = rmsprop_step(a, b, sa, sb, loss, lr, beta, eps)
    return a, b
test_rmsprop_method(rmsprop_method)

se RMSprop animeret med jeres egen `rmsprop_skridt` kaldt for hvert billede: venstre panel er den rå gradient, midten er det SAMME rum omskaleret med $\sqrt{s_t}$ (en aflang dal bliver rund), højre er skridtet der rent faktisk tages:

In [ ]:
landscape = [l for l in tk.LANDSCAPES if l["name"] == "linjefitting"][0]
lr, beta, eps = 0.02, 0.1, 1e-8   # I har selv valgt disse (ikke rmsprop_metode's defaults)
show_scale = 0.01   # visningsskala for den RÅ gradient-pil (den rmsprop-skalerede pil er allerede ca. i størrelsesorden 1)
color_range = sesy_viz.loss_range(landscape["loss"], landscape["a_range"], landscape["b_range"])   # samme farveskala i alle paneler/billeder
a, b = 0, 0
sa, sb = 0.0, 0.0
path = [(a, b)]

frames = []
for _ in range(100):
    da, db = get_gradient(landscape["loss"], a, b)   # kun til visning i panel 1
    ny_a, ny_b, sa, sb = rmsprop_step(a, b, sa, sb, landscape["loss"], lr, beta, eps)   # jeres egen funktion
    scale_a, scale_b = sa**0.5 + eps, sb**0.5 + eps
    show_a, show_b = da / scale_a, db / scale_b   # den skalerede retning — det panel 2/3 viser

    # panel 2's a-akse beholder landskabets EGEN, faste range hele vejen igennem (aldrig
    # zoomet ind/ud pr. billede) — så I kan se rummet selv strække/klemme sig (aksetallene
    # ændrer sig med skala_a: store tal tidligt, tæt på 1 midtvejs, små sent). b-aksens
    # BREDDE sættes til at matche a-aksens (samme antal enheder i det omskalerede rum — et
    # ægte KVADRAT, ikke et skævt rektangel), men centreret om nuværende b (ikke om 0), så
    # det er dér kurven rent faktisk bøjer der vises.
    width = (landscape["a_range"][1] - landscape["a_range"][0]) * scale_a
    lokal_b_range = (b - width / (2 * scale_b), b + width / (2 * scale_b))

    shared = dict(path=list(path), colorbar=False, color_range=color_range)
    frames.append((
        sesy_viz.loss_contour(landscape["loss"], landscape["a_range"], landscape["b_range"], gradient=(show_scale * da, show_scale * db), title="gradient nu", **shared),
        sesy_viz.loss_contour(landscape["loss"], landscape["a_range"], lokal_b_range, scale=(scale_a, scale_b),
                              gradient=(show_scale * da, show_scale * db),
                              extra_arrows=[(a, b, show_a, show_b, "orange", 1.0)],
                              title="skaleret rum", **shared),
        sesy_viz.loss_contour(landscape["loss"], landscape["a_range"], landscape["b_range"], gradient=(show_a, show_b),
                              gradient_color="orange", title="rmsprop-skridt", **shared),
    ))
    a, b = ny_a, ny_b
    path.append((a, b))

sesy_viz.animate(frames, max_frames=50)

**Frit spil:** prøv jeres egen variant af RMSprop — fx et andet `beta`/`lr`, eller kombinér
med en idé fra momentum. Tilføj den til listen og sammenlign:

In [ ]:
def own_gd_method_v3(loss, start, lr=0.02, n=100):
    a, b = start
    da, db = get_gradient(loss, a, b)
    for _ in range(n):
        a, b = a - lr * da, b - lr * db
    return a, b

tk.evaluate_gd_method([
    ("gd", gd_method),
    ("momentum", momentum_method),
    ("rmsprop", rmsprop_method),
    ("egen v3", own_gd_method_v3),
])

## ADAM

Adam kombinerer de to metoder ovenfor: $v_t$ fra momentum og $s_t$ fra RMSprop.
I kan genbruger jeres `momentum_v_opdater` og `rmsprop_s_opdater` fra før.
plus en ny **bias-korrektion** af begge, vigtig i de første skridt, hvor $v_0=s_0=0$ ellers trækker
gennemsnittet kunstigt mod nul:

$$v_t = \beta_1 v_{t-1} + (1-\beta_1)\nabla L(a_t,b_t), \qquad s_t = \beta_2 s_{t-1} + (1-\beta_2)\nabla L(a_t,b_t)^2$$

$$\hat v_t = \frac{v_t}{1-\beta_1^t}, \qquad \hat s_t = \frac{s_t}{1-\beta_2^t}, \qquad (a,b)_{t+1} = (a,b)_t - \text{lr}\cdot\frac{\hat v_t}{\sqrt{\hat s_t}+\varepsilon}$$

In [ ]:
from regression2_test import *

# ét adam-skridt: samme v/s-opdatering som momentum/rmsprop (genbrugt!), plus bias-korrektion af begge
def adam_step(a, b, va, vb, sa, sb, t, loss, lr, beta1, beta2, eps):
    da, db = get_gradient(loss, a, b)
    va, vb = momentum_v_update(va, vb, da, db, beta1)
    sa, sb = rmsprop_s_update(sa, sb, da, db, beta2)
    vha, vhb = va / (1 - beta1**t), vb / (1 - beta1**t)
    sha, shb = sa / (1 - beta2**t), sb / (1 - beta2**t)
    return a - lr*vha/(sha**0.5+eps), b - lr*vhb/(shb**0.5+eps), va, vb, sa, sb
test_adam_step(adam_step)

# løkken udenom: kald adam_skridt n gange i træk - t starter ved 1 (bruges i bias-korrektionen)
def adam_method(loss, start, lr=0.2, beta1=0.9, beta2=0.99, eps=1e-8, n=99):
    a, b = start
    va, vb = 0.0, 0.0
    sa, sb = 0.0, 0.0
    for t in range(1, n + 1):
        a, b, va, vb, sa, sb = adam_step(a, b, va, vb, sa, sb, t, loss, lr, beta1, beta2, eps)
    return a, b
test_adam_method(adam_method)

# alt I har bygget i denne blok, samlet i ét sidste overblik:
tk.evaluate_gd_method([
    ("gd", gd_method),
    ("momentum", momentum_method),
    ("rmsprop", rmsprop_method),
    ("adam", adam_method),
])

se adam animeret, med jeres egen `adam_skridt` kaldt for hvert billede.
- panel 1 er den originale gradient
- panel 2 er $\hat v_t$ (momentum-viften, korrigeret), 
- panel 3 er samme $\hat v_t$ (orange) skaleret ned med $\sqrt{\hat s_t}$ til en ny, magenta pil - den pil er adam-skridtet i panel 4:

In [ ]:
landscape = [l for l in tk.LANDSCAPES if l["name"] == "rosenbrock"][0]
lr, beta1, beta2, eps = 0.2, 0.9, 0.99, 1e-8   # højere lr og lavere beta2 end adam_metode's
                                                # defaults — kun her, så begge korrektioner ses
                                                # tydeligt inden for 80 billeder
show_scale = 0.2   # visningsskala for den rå gradient/v̂-pil (v̂/ŝ er allerede ca. størrelsesorden 1)
color_range = sesy_viz.loss_range(landscape["loss"], landscape["a_range"], landscape["b_range"])
a, b = -1.0, -0.5
va, vb = 0.0, 0.0
sa, sb = 0.0, 0.0
history = []   # (da, db, ref_a, ref_b) — reference frosset fra dengang gradienten blev målt
measurements = []   # (a, b, ref_a, ref_b), til panel 1's grå pile
path = [(a, b)]

frames = []
for t in range(1, 101):
    da, db = get_gradient(landscape["loss"], a, b)   # kun til visning i panel 1
    korr1 = 1 / (1 - beta1**t)
    reference = (korr1 * show_scale * (1 - beta1) * da, korr1 * show_scale * (1 - beta1) * db)
    history.append((da, db, *reference))
    measurements.append((a, b, *reference))

    ny_a, ny_b, va, vb, sa, sb = adam_step(a, b, va, vb, sa, sb, t, landscape["loss"], lr, beta1, beta2, eps)   # jeres egen funktion
    vha, vhb = va / (1 - beta1**t), vb / (1 - beta1**t)
    scale_a, scale_b = (sa / (1 - beta2**t))**0.5 + eps, (sb / (1 - beta2**t))**0.5 + eps

    # samme vifte-konstruktion som momentum: reference er frosset (med SIN egen korr1 fra
    # dengang) — kun "nu"-bidraget bruger dagens korr1
    n = len(history)
    pair = [((ref_a, ref_b), (korr1 * show_scale * (1 - beta1) * beta1**alder * hda, korr1 * show_scale * (1 - beta1) * beta1**alder * hdb))
           for alder, (hda, hdb, ref_a, ref_b) in zip(range(n - 1, -1, -1), history)]

    v_show_a, v_show_b = show_scale * vha, show_scale * vhb   # v̂ (orange) — samlet i panel 2, bæres videre til panel 3
    sv_show_a, sv_show_b = vha / scale_a, vhb / scale_b     # v̂/ŝ (magenta) — nyt i panel 3, bæres videre til panel 4

    # a-aksen beholder landskabets EGEN, faste range hele vejen (samme idé som RMSprop-
    # panelet ovenfor) — b-aksens bredde matcher a's (et ægte kvadrat), centreret om
    # nuværende b.
    width = (landscape["a_range"][1] - landscape["a_range"][0]) * scale_a
    lokal_b_range = (b - width / (2 * scale_b), b + width / (2 * scale_b))

    shared = dict(path=list(path), colorbar=False)
    frames.append((
        sesy_viz.loss_contour(landscape["loss"], landscape["a_range"], landscape["b_range"], gradient=(show_scale * da, show_scale * db),
                              extra_arrows=[(ma, mb, mda, mdb, "gray", 0.5) for ma, mb, mda, mdb in measurements],
                              title="gradient nu (+ tidligere målinger)", color_range=color_range, **shared),
        sesy_viz.vector_fan(pair, sum_vector=(v_show_a, v_show_b), title="v (korrigeret): momentum-vifte"),
        sesy_viz.loss_contour(landscape["loss"], landscape["a_range"], lokal_b_range, scale=(scale_a, scale_b),
                              gradient=(v_show_a, v_show_b), gradient_color="orange",
                              extra_arrows=[(a, b, sv_show_a, sv_show_b, "magenta", 1.0)],
                              title="s (korrigeret): skaleret rum", color_range=color_range, **shared),
        sesy_viz.loss_contour(landscape["loss"], landscape["a_range"], landscape["b_range"],
                              gradient=(sv_show_a, sv_show_b), gradient_color="magenta", title="adam-skridt", color_range=color_range, **shared),
    ))
    a, b = ny_a, ny_b
    path.append((a, b))

sesy_viz.animate(frames, max_frames=50)

**Sidste frit spil:** byg jeres bedste mulige metode — kombinér gerne idéer fra gd/momentum/
rmsprop/adam, eller prøv noget helt nyt. Tilføj den til listen sammen med alle de andre, og se
om I kan slå dem alle:

In [ ]:
def own_best_method(loss, start, lr=0.01, n=100):
    a, b = start
    for _ in range(n):
        a, b = gd_step(a, b, loss, lr)
    return a, b

tk.evaluate_gd_method([
    ("gd", gd_method),
    ("momentum", momentum_method),
    ("rmsprop", rmsprop_method),
    ("adam", adam_method),
    ("min bedste", own_best_method),
])